# vector_checker — разбор расхождений service ↔ 1С

Тетрадка вместо чтения JSON: сводка, таблица cell-mismatch, drill-down по `LOSS_NUMBER`.

**Режимы**
- `USE_SYNTHETIC = True` — демо на синтетике (match / mismatch)
- `False` — реальный `df_for_service` + Excel из `work/excel/`

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

# Ищем examples/querulus независимо от cwd ядра
def _find_querulus_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "integration" / "tests" / "vector_checker").is_dir() and (
            cand / "integration" / "main.py"
        ).is_file():
            return cand
    raise FileNotFoundError("Не найден examples/querulus (нужен integration/main.py)")

QUERULUS_ROOT = _find_querulus_root()
if str(QUERULUS_ROOT) not in sys.path:
    sys.path.insert(0, str(QUERULUS_ROOT))

from integration.tests.vector_checker.compare_features import (
    compare_feature_frames,
    load_excel_export,
    mismatch_feature_counts,
    report_to_frame,
    side_by_side_for_loss,
)
from integration.tests.vector_checker.paths import (
    EXCEL_DIR,
    LOSS_NUMBER_COL,
    ensure_work_dirs,
    resolve_df_for_service,
)
from integration.tests.vector_checker.query_inject import sample_loss_numbers
from integration.tests.vector_checker.synthetic import write_synthetic_fixtures

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

ensure_work_dirs()
print("QUERULUS_ROOT:", QUERULUS_ROOT)
print("EXCEL_DIR:", EXCEL_DIR)

QUERULUS_ROOT: C:\Users\grchi\Desktop\pet_projects\outboxml\examples\querulus
EXCEL_DIR: C:\Users\grchi\Desktop\pet_projects\outboxml\examples\querulus\integration\tests\vector_checker\work\excel


## 1. Источники

- `USE_SYNTHETIC=True` + `SYNTHETIC_MISMATCH=True` — заранее сломанный Excel (удобно смотреть FAIL)
- для боя: положи выгрузку 1С в `work/excel/` и укажи имя файла
- `N_LOSSES` / `RANDOM_SEED` — случайная выборка убытков из service (как `prepare -n --seed`)

In [2]:
USE_SYNTHETIC = True
SYNTHETIC_MISMATCH = True  # True → export_1c_mismatch.xlsx; False → export_1c_ok.xlsx

# Выборка убытков из service (None = все)
N_LOSSES: int | None = None
RANDOM_SEED = 42

# Боевой режим (USE_SYNTHETIC=False):
EXCEL_NAME = "export.xlsx"  # файл в work/excel/
DF_PATH: str | None = None  # None → автопоиск df_for_service

if USE_SYNTHETIC:
    paths = write_synthetic_fixtures()
    df_path = paths["df"]
    excel_path = paths["excel_bad" if SYNTHETIC_MISMATCH else "excel_ok"]
else:
    df_path = resolve_df_for_service(DF_PATH)
    excel_path = EXCEL_DIR / EXCEL_NAME
    if not excel_path.is_file():
        raise FileNotFoundError(
            f"Нет Excel: {excel_path}. Положи выгрузку 1С в {EXCEL_DIR}"
        )

print("df   :", df_path)
print("excel:", excel_path)
print("sample:", N_LOSSES, "seed:", RANDOM_SEED)

df   : C:\Users\grchi\Desktop\pet_projects\outboxml\examples\querulus\integration\tests\vector_checker\fixtures\synthetic\df_for_service.parquet
excel: C:\Users\grchi\Desktop\pet_projects\outboxml\examples\querulus\integration\tests\vector_checker\fixtures\synthetic\export_1c_mismatch.xlsx


In [3]:
service_full = pd.read_parquet(df_path)
excel = load_excel_export(excel_path)

sampled = sample_loss_numbers(
    service_full[LOSS_NUMBER_COL].tolist(),
    N_LOSSES,
    random_seed=RANDOM_SEED,
)
_keys = set(sampled)
service = service_full.loc[
    service_full[LOSS_NUMBER_COL].map(lambda x: str(x).strip() if pd.notna(x) else "").isin(_keys)
].copy()
# Excel тоже сужаем до выборки (лишние строки из полной выгрузки не шумят)
if LOSS_NUMBER_COL in excel.columns:
    excel = excel.loc[
        excel[LOSS_NUMBER_COL].map(lambda x: str(x).strip() if pd.notna(x) else "").isin(_keys)
    ].copy()

print("service:", service.shape, f"(из {service_full.shape[0]}, n={N_LOSSES}, seed={RANDOM_SEED})")
print("excel:", excel.shape)
display(service.head(3))
display(excel.head(3))

service: (5, 24) excel: (4, 20)


,LOSS_NUMBER,INCIDENT_NUMBER,FILIAL,EVENT_CREATED_BY_GIBDD_FLAG,VICTIM_MAX_WEIGHT,RECIEVE_METHOD,APPLICANT_FORM,APPLICANT_AGE,VICTIM_VEHICLE_CATEGORY,GUILTY_CAPACITY_ENGINE,...,VICTIM_VEHICLE_COUNTRY,APPLY_DELAY,EVENT_DATE,PAYMENT_ORDER_DATE_TIME,VICTIM_VEHICLE_BRAND,VALUE_BEFORE_WITH,VALUE_BEFORE_WITHOUT,REGION,preds_cf,preds_rg
0,10000000,10000000,Белгородский,1,1500,SMS,Потерпевший,30,Эконом 0-6,1.2,...,Россия,3,2025-01-15T10:00:00,2025-01-20T00:00:00,Toyota,40000,45000,77,0.0,0.0
1,10000001,10000001,ВСК-Москва,0,1510,Email,Представитель (не автоюрист),31,Средний 7+,1.4,...,Россия,4,2025-02-15T10:00:00,2025-02-20T00:00:00,Mazda,40001,45001,77,0.1,1000.0
2,10000002,10000002,Краснодарский,1,1520,Лично,Выгодоприобретатель,32,Премиум 7+,1.6,...,Россия,5,2025-03-15T10:00:00,2025-03-20T00:00:00,Skoda,40002,45002,77,0.2,2000.0


,LOSS_NUMBER,INCIDENT_NUMBER,FILIAL,EVENT_CREATED_BY_GIBDD_FLAG,VICTIM_MAX_WEIGHT,RECIEVE_METHOD,APPLICANT_FORM,APPLICANT_AGE,VICTIM_VEHICLE_CATEGORY,GUILTY_CAPACITY_ENGINE,VICTIM_VEHICLE_AGE,EVENT_YEAR,AMOUNT_REPAIR,LOSS_UNIT_ZONE,EVENT_DATE,PAYMENT_ORDER_DATE_TIME,VICTIM_VEHICLE_BRAND,VALUE_BEFORE_WITH,VALUE_BEFORE_WITHOUT,REGION
0,10000000,10000000,ЛОМАЕМ_СОВПАДЕНИЕ,1,1500,SMS,Потерпевший,30,Эконом 0-6,1.2,2,2024,50000,Зона тестовая 0,2025-01-15T10:00:00,2025-01-20T00:00:00,Toyota,40000,45000,77
1,10000002,10000002,Краснодарский,1,1520,Лично,Выгодоприобретатель,32,Премиум 7+,1.6,4,2024,52000,Зона тестовая 2,2025-03-15T10:00:00,2025-03-20T00:00:00,Skoda,40002,45002,77
2,10000003,10000003,Казанский,0,1530,Не оповещать,Представитель (по доверенности),33,Иное,1.8,5,2025,53000,Зона тестовая 3,2025-04-15T10:00:00,2025-04-20T00:00:00,Lexus,40003,45003,77


## 2. Сводка и клеточные расхождения

`mismatches` — длинная таблица: одна строка = один `LOSS_NUMBER` × одна фича.

In [4]:
report, mismatches = compare_feature_frames(service, excel)

display(report_to_frame(report))
print("ok:", report.ok)
print("нет в Excel:", report.features_missing_in_excel)
print("нет в service:", report.features_missing_in_service)

display(mismatch_feature_counts(mismatches))
display(mismatches)

,ok,n_service,n_excel,n_matched_keys,n_only_service,n_only_excel,n_mismatch_cells,n_features_compared,n_missing_in_excel,n_missing_in_service
0,False,5,4,4,1,0,1,19,2,0


ok: False
нет в Excel: ['VICTIM_VEHICLE_COUNTRY', 'APPLY_DELAY']
нет в service: []


,feature,n_mismatch
0,FILIAL,1


,LOSS_NUMBER,feature,service,excel
0,10000000,FILIAL,Белгородский,ЛОМАЕМ_СОВПАДЕНИЕ


## 3. Ключи только в одной стороне

In [5]:
only_service = pd.DataFrame({"LOSS_NUMBER": report.only_service_losses})
only_excel = pd.DataFrame({"LOSS_NUMBER": report.only_excel_losses})

print("только в service:", len(only_service))
display(only_service.head(20))
print("только в excel:", len(only_excel))
display(only_excel.head(20))

if len(only_service):
    display(service.loc[service["LOSS_NUMBER"].astype(str).isin(only_service["LOSS_NUMBER"])].head())

только в service: 1


,LOSS_NUMBER
0,10000001


только в excel: 0


,LOSS_NUMBER


,LOSS_NUMBER,INCIDENT_NUMBER,FILIAL,EVENT_CREATED_BY_GIBDD_FLAG,VICTIM_MAX_WEIGHT,RECIEVE_METHOD,APPLICANT_FORM,APPLICANT_AGE,VICTIM_VEHICLE_CATEGORY,GUILTY_CAPACITY_ENGINE,...,VICTIM_VEHICLE_COUNTRY,APPLY_DELAY,EVENT_DATE,PAYMENT_ORDER_DATE_TIME,VICTIM_VEHICLE_BRAND,VALUE_BEFORE_WITH,VALUE_BEFORE_WITHOUT,REGION,preds_cf,preds_rg
1,10000001,10000001,ВСК-Москва,0,1510,Email,Представитель (не автоюрист),31,Средний 7+,1.4,...,Россия,4,2025-02-15T10:00:00,2025-02-20T00:00:00,Mazda,40001,45001,77,0.1,1000.0


## 4. Drill-down по одному убытку

Подставь `LOSS_NUMBER` из `mismatches` или `only_service` — увидишь side-by-side по всем общим фичам.

In [6]:
if not mismatches.empty:
    LOSS_TO_INSPECT = mismatches["LOSS_NUMBER"].iloc[0]
elif report.only_service_losses:
    LOSS_TO_INSPECT = report.only_service_losses[0]
else:
    LOSS_TO_INSPECT = str(service["LOSS_NUMBER"].iloc[0])

print("LOSS_TO_INSPECT:", LOSS_TO_INSPECT)
sbs = side_by_side_for_loss(service, excel, LOSS_TO_INSPECT)
display(sbs.loc[~sbs["match"]] if (~sbs["match"]).any() else sbs)
display(sbs)  # полный вектор рядом

LOSS_TO_INSPECT: 10000000


,feature,service,excel,match
6,FILIAL,Белгородский,ЛОМАЕМ_СОВПАДЕНИЕ,False


,feature,service,excel,match
0,AMOUNT_REPAIR,50000,50000,True
1,APPLICANT_AGE,30,30,True
2,APPLICANT_FORM,Потерпевший,Потерпевший,True
3,EVENT_CREATED_BY_GIBDD_FLAG,1,1,True
4,EVENT_DATE,2025-01-15T10:00:00,2025-01-15T10:00:00,True
5,EVENT_YEAR,2024,2024,True
6,FILIAL,Белгородский,ЛОМАЕМ_СОВПАДЕНИЕ,False
7,GUILTY_CAPACITY_ENGINE,1.2,1.2,True
8,INCIDENT_NUMBER,10000000,10000000,True
9,LOSS_UNIT_ZONE,Зона тестовая 0,Зона тестовая 0,True


## 5. Фильтр по фиче

Например: все убытки, где разъехался `FILIAL`.

In [7]:
FEATURE = "FILIAL"  # поменяй на нужную

if mismatches.empty:
    print("Расхождений по ячейкам нет")
else:
    subset = mismatches.loc[mismatches["feature"] == FEATURE]
    display(subset)
    if len(subset):
        keys = subset["LOSS_NUMBER"].astype(str)
        cols = ["LOSS_NUMBER", FEATURE] if FEATURE in service.columns else ["LOSS_NUMBER"]
        left = service.assign(_k=service["LOSS_NUMBER"].astype(str)).loc[
            lambda d: d["_k"].isin(keys), cols
        ].rename(columns={FEATURE: f"{FEATURE}__service"} if FEATURE in cols else {})
        right_cols = [c for c in ["LOSS_NUMBER", FEATURE] if c in excel.columns]
        right = excel.assign(_k=excel["LOSS_NUMBER"].astype(str)).loc[
            lambda d: d["_k"].isin(keys), right_cols
        ].rename(columns={FEATURE: f"{FEATURE}__excel"} if FEATURE in right_cols else {})
        display(
            left.merge(
                right,
                on="LOSS_NUMBER",
                how="outer",
                suffixes=("_svc", "_xl"),
            )
        )

,LOSS_NUMBER,feature,service,excel
0,10000000,FILIAL,Белгородский,ЛОМАЕМ_СОВПАДЕНИЕ


,LOSS_NUMBER,FILIAL__service,FILIAL__excel
0,10000000,Белгородский,ЛОМАЕМ_СОВПАДЕНИЕ
